# Corrected WESAD suite on Colab
Upload codes/wesad_corrected.py, codes/wesad_suite.py and all 15 datasets/WESAD/S*.pkl files into MyDrive/WESAD_Paper1_GPU. Use a fresh results folder. Select a GPU runtime first. This retains recurrent_dropout=0.3 and the existing architecture; GPU acceleration must be measured. No CPU results are merged automatically. Completed fits resume; an interrupted fit restarts. Results/checkpoints live on Drive. Allow substantial Drive space for the 12.88 GiB raw dataset plus checkpoints.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
PROJECT = Path('/content/drive/MyDrive/WESAD_Paper1_GPU')
assert (PROJECT / 'codes' / 'wesad_corrected.py').is_file()
assert (PROJECT / 'codes' / 'wesad_suite.py').is_file()
expected = {2,3,4,5,6,7,8,9,10,11,13,14,15,16,17}
actual = {int(p.stem[1:]) for p in (PROJECT / 'datasets' / 'WESAD').glob('S*.pkl')}
assert actual == expected, (actual, expected)
print('All 15 subject files and both code files found.')

In [ ]:
%pip install -q scipy scikit-learn matplotlib

In [ ]:
import tensorflow as tf
import sys, subprocess, json
print('Python:', sys.version)
print('TensorFlow:', tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print('GPUs:', gpus)
assert gpus, 'Select Runtime > Change runtime type > GPU and reconnect before training.'
OUT = PROJECT / 'results' / 'wesad_complete'
OUT.mkdir(parents=True, exist_ok=True)
(OUT / 'colab_environment.txt').write_text(subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True))

In [ ]:
# Validate/cache all subjects and generate the subject split plan. No training yet.
subprocess.run([sys.executable, '-u', str(PROJECT / 'codes' / 'wesad_suite.py'), '--prepare-only'], cwd=PROJECT, check=True)
print((OUT / 'data_audit.json').read_text())

In [ ]:
# Long-running foreground training. Completed fits are skipped after reconnecting.
# Do not run a second worker against this same Drive output folder.
subprocess.run([sys.executable, '-u', str(PROJECT / 'codes' / 'wesad_suite.py')], cwd=PROJECT, check=True)

In [ ]:
print((OUT / 'status.json').read_text())
if (OUT / 'RESULTS.md').exists():
    from IPython.display import display, Markdown
    display(Markdown((OUT / 'RESULTS.md').read_text()))
if (OUT / 'validation.json').exists():
    print((OUT / 'validation.json').read_text())